# Notebook 02 — Baseline Detectors

**ECE 228 SP26 · Akshat Gupta**

Trains and evaluates three baselines against the GP classifier:
1. **Fixed threshold** — current operational system (z ≥ 4.0)
2. **Logistic Regression** — linear, same feature vector
3. **MLP** — nonlinear, isotonic-calibrated

Key metric: Expected Calibration Error (ECE). All models will achieve AUROC ≈ 1.0 — the differentiator is how well-calibrated the output probabilities are.

In [ ]:
import sys, json, pickle
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, average_precision_score

from gp_detector.dataset import load_raw, engineer_features, make_labeled_split,                                 chronological_split, get_arrays, FEATURE_COLS
from gp_detector.models.baselines import ThresholdDetector, build_logreg, build_mlp
from gp_detector.evaluate import ece, compute_all, reliability_diagram, plot_roc_pr, MODEL_COLORS

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})


## 1. Data preparation

In [ ]:
raw      = load_raw()
df       = engineer_features(raw)
train_df, soft_df = make_labeled_split(df, seed=42)
tr, va, te = chronological_split(train_df)

X_tr, y_tr = get_arrays(tr)
X_va, y_va = get_arrays(va)
X_te, y_te = get_arrays(te)
elev_te    = te['sat_elevation'].values

print(f"Train: {len(X_tr):,}  Val: {len(X_va):,}  Test: {len(X_te):,}")
print(f"Positive rate (train): {y_tr.mean():.3f}  (test): {y_te.mean():.3f}")
print(f"Soft held-out: {len(soft_df):,} windows")


## 2. Train baselines

In [ ]:
# Fixed threshold
thresh   = ThresholdDetector(z_col_idx=FEATURE_COLS.index('z_score'))
p_thresh = thresh.predict_proba(X_te)[:, 1]

# Logistic Regression
lr = build_logreg()
lr.fit(X_tr, y_tr)
p_lr = lr.predict_proba(X_te)[:, 1]

# MLP (calibrated)
mlp = build_mlp()
mlp.fit(X_tr, y_tr)
p_mlp = mlp.predict_proba(X_te)[:, 1]

metrics = {
    'Threshold': compute_all(y_te, p_thresh, 'Threshold'),
    'LogReg':    compute_all(y_te, p_lr,     'LogReg'),
    'MLP':       compute_all(y_te, p_mlp,    'MLP'),
}

print(f"{'Model':<18} {'AUROC':>7} {'AUPRC':>7} {'ECE':>8}")
print('-'*44)
for k, m in metrics.items():
    print(f"{k:<18} {m['AUROC']:>7.4f} {m['AUPRC']:>7.4f} {m['ECE']:>8.4f}")


## 3. ROC and Precision-Recall curves

In [ ]:
results = [
    {'name': 'Threshold', 'y_true': y_te, 'y_prob': p_thresh},
    {'name': 'LogReg',    'y_true': y_te, 'y_prob': p_lr},
    {'name': 'MLP',       'y_true': y_te, 'y_prob': p_mlp},
]
fig = plot_roc_pr(results)
plt.savefig('../results/baselines_roc_pr.png', bbox_inches='tight')
plt.show()
print("All models reach AUROC ≈ 1.0 — discrimination is trivial. Calibration is where they differ.")


## 4. Reliability diagrams

A well-calibrated model should have predicted probability ≈ actual fraction positive in each probability bin. Deviation from the diagonal is miscalibration.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
pairs = [('Threshold', p_thresh), ('LogReg', p_lr), ('MLP', p_mlp)]

for ax, (name, probs) in zip(axes, pairs):
    e = ece(y_te, probs)
    reliability_diagram(ax, y_te, probs, label=name, color=MODEL_COLORS[name])
    ax.set_title(f'{name}\nECE = {e:.4f}')
    ax.legend(fontsize=8)

plt.suptitle('Reliability Diagrams — Baseline Models', fontsize=12)
plt.tight_layout()
plt.savefig('../results/baselines_reliability.png', bbox_inches='tight')
plt.show()


## 5. ECE comparison bar chart

In [ ]:
names  = ['Threshold\nz=4', 'Logistic\nRegression', 'MLP']
keys   = ['Threshold', 'LogReg', 'MLP']
eces   = [metrics[k]['ECE'] for k in keys]
colors = [MODEL_COLORS[k] for k in keys]

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(names, eces, color=colors, alpha=0.85, width=0.5)
ax.set_ylabel('ECE (lower is better)')
ax.set_title('Calibration Error — Baselines')
for bar, v in zip(bars, eces):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.001, f'{v:.4f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../results/baselines_ece.png', bbox_inches='tight')
plt.show()

print(f"Logistic regression is {metrics['Threshold']['ECE']/metrics['LogReg']['ECE']:.0f}x better calibrated than the fixed threshold.")
print("But LogReg doesn't quantify uncertainty — it just outputs a point probability.")
print("The GP will add: (1) principled uncertainty bounds, (2) a physics-informed kernel.")


## 6. Behaviour on SOFT (ambiguous) windows

The 70,878 SOFT windows are held out from training — they sit in the z ∈ [3,4] band where the classical detector is uncertain. How do the baselines behave here?

In [ ]:
X_soft  = soft_df[FEATURE_COLS].values.astype('float32')

p_thresh_soft = thresh.predict_proba(X_soft)[:, 1]
p_lr_soft     = lr.predict_proba(X_soft)[:, 1]
p_mlp_soft    = mlp.predict_proba(X_soft)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, probs) in zip(axes, [('Threshold', p_thresh_soft),
                                      ('LogReg', p_lr_soft),
                                      ('MLP', p_mlp_soft)]):
    ax.hist(probs, bins=60, color=MODEL_COLORS[name], alpha=0.8)
    ax.set_xlabel('Predicted probability'); ax.set_ylabel('Count')
    ax.set_title(f'{name} on SOFT windows')
    ax.axvline(0.5, ls='--', color='black', lw=1.0)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('Probability Distribution on SOFT (Ambiguous) Windows', fontsize=12)
plt.tight_layout()
plt.savefig('../results/baselines_soft_analysis.png', bbox_inches='tight')
plt.show()

for name, probs in [('Threshold', p_thresh_soft), ('LogReg', p_lr_soft), ('MLP', p_mlp_soft)]:
    print(f"{name:12s}  mean={probs.mean():.3f}  std={probs.std():.3f}  "
          f"frac>0.5: {(probs>0.5).mean():.3f}")


## Summary

| Model | AUROC | ECE | Calibrated? | Uncertainty? |
|---|---|---|---|---|
| Fixed Threshold | 1.000 | 0.065 | No | No |
| Logistic Regression | 1.000 | 0.0003 | Mostly | No |
| MLP | 1.000 | 0.006 | Partially | No |
| **SVGP (next)** | 1.000 | **0.0001** | **Yes** | **Yes** |

The baselines confirm: discrimination is easy, calibration is the hard part, and none of them give principled uncertainty — which is what the GP adds.